In [1]:
import numpy as np
import cv2
import librosa
import soundfile as sf

# ============ NEW: pixel-to-multiple-frequencies mapping into 1-4kHz, source/destination resolution decoupled ============
def frequency_remap_hires(spectrogram, sr=22050, n_fft_out=4096, target_low=1000.0, target_high=4000.0):
    """
    Same log curve as frequency_remap (f_new = a*log(f_old+1)+b), but source
    and destination axes are no longer forced to be the same size.

    - spectrogram keeps its own row count (513, the image's own resolution -
      unrelated to n_fft now).
    - the OUTPUT gets n_fft_out//2+1 rows instead, so the 1-4kHz band
      actually has room: ~557 bins at n_fft_out=4096, vs ~139 bins at the
      original n_fft=1024.
    - forward push + midpoint ownership: every source row contributes to
      SOME destination bin (nothing silently skipped, unlike frequency_remap's
      backward/pull approach).

    Allocation across the band is still UNEVEN (the curve itself hands the
    low end of 1-4kHz to a tiny sliver of source frequencies and crowds
    almost everything else into the upper end) - n_fft_out gives more total
    room to work with, it doesn't flatten the curve itself.
    """
    n_src, n_frames = spectrogram.shape
    n_bins_out = n_fft_out // 2 + 1

    freq_axis_src = np.linspace(0, sr / 2, n_src)

    b = target_low
    a = (target_high - target_low) / np.log(freq_axis_src[-1] + 1)

    f_new = a * np.log(freq_axis_src + 1) + b        # target Hz for each source row
    dst_pos = f_new / (sr / n_fft_out)                # target Hz -> destination bin index

    boundaries = (dst_pos[:-1] + dst_pos[1:]) / 2.0   # midpoint between neighboring source rows' targets

    remapped = np.zeros((n_bins_out, n_frames), dtype=spectrogram.dtype)
    freq_axis_out = np.linspace(0, sr / 2, n_bins_out)
    in_band = (freq_axis_out >= target_low) & (freq_axis_out <= target_high)

    for d in np.where(in_band)[0]:
        src_lo, src_hi = np.searchsorted(boundaries, [d - 0.5, d + 0.5])
        src_lo = min(src_lo, n_src - 1)
        src_hi = min(max(src_hi, src_lo + 1), n_src)
        remapped[d, :] = spectrogram[src_lo:src_hi, :].mean(axis=0)

    return remapped, a, b
# ============ END NEW ============


def image_to_audio_multi_freq(image_path, output_audio="output.wav", sr=22050, n_fft_out=4096):
    # Load image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError("Image not found or invalid path.")

    # Resize image - height stays at 513 regardless of n_fft_out; this is the
    # SOURCE resolution (how much real image detail we have), decoupled from
    # the DESTINATION resolution (n_fft_out) below.
    target_height = 513
    target_width = 128
    image = cv2.resize(image, (target_width, target_height))

    # Normalize to [0, 1] and flip vertically (spectrograms use bottom-to-top frequency)
    image_normalized = np.flipud(image.astype(np.float32) / 255.0)

    # Convert image to dB-scaled "spectrogram" (add epsilon to avoid log(0))
    spectrogram_db = librosa.amplitude_to_db(image_normalized + 1e-7, ref=np.max)

    # Convert dB back to linear amplitude
    spectrogram_linear = librosa.db_to_amplitude(spectrogram_db)

    # ============ NEW: remap into 1-4kHz, output sized by n_fft_out (not by the source's 513 rows) ============
    spectrogram_linear, a, b = frequency_remap_hires(spectrogram_linear, sr=sr, n_fft_out=n_fft_out)
    print(f"  remap params -> a={a:.2f}, b={b:.2f}, output bins={spectrogram_linear.shape[0]}")
    # ============ END NEW ============

    # Griffin-Lim parameters - driven by n_fft_out now, same 50% overlap ratio as your original 512/1024
    n_fft = n_fft_out
    hop_length = n_fft_out // 2
    win_length = n_fft_out

    # Reconstruct audio
    audio = librosa.griffinlim(
        spectrogram_linear,
        n_iter=32,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length
    )

    # Save as WAV file
    sf.write(output_audio, audio, sr)
    print(f"Audio saved to {output_audio}")

# Example usage
image_to_audio_multi_freq(r"D:\Documents\Iquisitionis\103\vehicle-type-recognition\Dataset\Car\Image_32.jpg", "car_1908.wav")

  remap params -> a=322.30, b=1000.00, output bins=2049
Audio saved to car_1908.wav
